# ============================================================
# STEP 8 — REGENERATE ALL FOLD PLOTS (no re-training)
# ============================================================
# All .npy prediction files already exist from the full CV run.
# This script loads them and generates all detailed plots for
# EVERY fold (not just fold 0), without touching the models.
#
# Runtime: ~5–10 minutes (plotting only, no training)
# ============================================================

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import (
    confusion_matrix, roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, det_curve, matthews_corrcoef,
    f1_score, cohen_kappa_score)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIGURATION — match your main eval script exactly
# ============================================================
DATA_DIR   = "/content/drive/MyDrive/TUMC_dataset"
TASKS      = ["5class", "binary"]
ALL_FOLDS  = list(range(5))
FN_COST, FP_COST = 10, 1

MODEL_ORDER = ["HistGB","LightGBM","XGBoost","RandomForest",
               "ExtraTrees","LogisticRegression","ComplementNB"]
MODEL_COLORS = {
    "HistGB":"#1565C0","LightGBM":"#1E88E5","XGBoost":"#42A5F5",
    "RandomForest":"#2E7D32","ExtraTrees":"#66BB6A",
    "LogisticRegression":"#F57F17","ComplementNB":"#B71C1C",
}

plt.rcParams.update({
    "font.family":"serif","font.size":10,"axes.labelsize":10,
    "axes.titlesize":11,"legend.fontsize":8.5,"figure.dpi":150,
    "savefig.dpi":600,"pdf.fonttype":42,"ps.fonttype":42,
    "lines.linewidth":1.6,"axes.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3,"grid.linestyle":"--",
})

# ============================================================
# 1. HELPERS
# ============================================================
def safe_name(s):
    for c in " /\\:()": s = s.replace(c,"_")
    return s

def makedirs(*paths):
    for p in paths: os.makedirs(p, exist_ok=True)

def save_fig(fig, *paths):
    for p in paths:
        ext = os.path.splitext(p)[1].lower()
        fig.savefig(p, dpi=600 if ext==".png" else None, bbox_inches="tight")
    plt.close(fig)

def load_fold_preds(csv_dir):
    """Load predictions and probabilities for all models in one fold."""
    data = {}
    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        ft  = os.path.join(csv_dir, f"y_true_{mid}.npy")
        fp  = os.path.join(csv_dir, f"y_pred_{mid}.npy")
        fpr = os.path.join(csv_dir, f"y_proba_{mid}.npy")
        if not (os.path.exists(ft) and os.path.exists(fp)):
            continue
        data[mn] = {
            "y_true":  np.load(ft),
            "y_pred":  np.load(fp),
            "y_proba": np.load(fpr) if os.path.exists(fpr) else None,
        }
    return data

# ============================================================
# 2. PLOT FUNCTIONS (fold-aware versions)
# ============================================================

def plot_roc(data, task, classes, class_names, is_bin, fold_id, png_dir, pdf_dir):
    n_sub = 1 if is_bin else len(classes)
    fig, axes = plt.subplots(1, n_sub, figsize=(6.5 if is_bin else 6*n_sub, 5.5))
    if is_bin: axes = [axes]
    for mn, d in data.items():
        y_proba = d["y_proba"]; y_true = d["y_true"]
        if y_proba is None: continue
        col = MODEL_COLORS.get(mn,"#607D8B")
        if is_bin:
            try:
                fpr,tpr,_ = roc_curve(y_true, y_proba[:,1])
                auc = roc_auc_score(y_true, y_proba[:,1])
                axes[0].plot(fpr,tpr,color=col,label=f"{mn} (AUC={auc:.3f})")
            except: pass
        else:
            yb = label_binarize(y_true, classes=classes)
            for ci,(ax,cn) in enumerate(zip(axes,class_names)):
                try:
                    fpr,tpr,_ = roc_curve(yb[:,ci], y_proba[:,ci])
                    auc = roc_auc_score(yb[:,ci], y_proba[:,ci])
                    ax.plot(fpr,tpr,color=col,label=f"{mn} ({auc:.3f})")
                    ax.set_title(cn, fontweight="bold")
                except: pass
    for ax in axes:
        ax.plot([0,1],[0,1],"k--",lw=0.8)
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=7)
    axes[0].set_title(f"ROC — {task} fold {fold_id}", fontweight="bold")
    fig.suptitle("Receiver Operating Characteristic", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,f"roc_curves.png"),
                  os.path.join(pdf_dir,f"roc_curves.pdf"))

def plot_pr(data, task, classes, class_names, is_bin, fold_id, png_dir, pdf_dir):
    n_sub = 1 if is_bin else len(classes)
    fig, axes = plt.subplots(1,n_sub,figsize=(6.5 if is_bin else 6*n_sub, 5.5))
    if is_bin: axes=[axes]
    for mn,d in data.items():
        y_proba=d["y_proba"]; y_true=d["y_true"]
        if y_proba is None: continue
        col=MODEL_COLORS.get(mn,"#607D8B")
        if is_bin:
            try:
                prec,rec,_=precision_recall_curve(y_true,y_proba[:,1])
                ap=average_precision_score(y_true,y_proba[:,1])
                axes[0].plot(rec,prec,color=col,label=f"{mn} (AP={ap:.3f})")
            except: pass
        else:
            yb=label_binarize(y_true,classes=classes)
            for ci,(ax,cn) in enumerate(zip(axes,class_names)):
                try:
                    prec,rec,_=precision_recall_curve(yb[:,ci],y_proba[:,ci])
                    ap=average_precision_score(yb[:,ci],y_proba[:,ci])
                    ax.plot(rec,prec,color=col,label=f"{mn} ({ap:.3f})")
                    ax.axhline(yb[:,ci].mean(),ls=":",color=col,lw=0.8)
                    ax.set_title(cn,fontweight="bold")
                except: pass
    for ax in axes:
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(fontsize=7)
    axes[0].set_title(f"PR — {task} fold {fold_id}",fontweight="bold")
    fig.suptitle("Precision-Recall (critical for imbalanced classes)",fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"pr_curves.png"),
                  os.path.join(pdf_dir,"pr_curves.pdf"))

def plot_det(data, task, is_bin, fold_id, png_dir, pdf_dir):
    if not is_bin: return
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    for mn,d in data.items():
        y_proba=d["y_proba"]; y_true=d["y_true"]
        if y_proba is None: continue
        try:
            fpr,fnr,_=det_curve(y_true,y_proba[:,1])
            ax.plot(fpr,fnr,color=MODEL_COLORS.get(mn,"#607D8B"),label=mn)
        except: pass
    ax.set_xlabel("FPR"); ax.set_ylabel("FNR")
    ax.set_title(f"DET — fold {fold_id}",fontweight="bold")
    ax.legend(); ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.4)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"det_curves.png"),
                  os.path.join(pdf_dir,"det_curves.pdf"))

def plot_confusion_all(data, task, classes, class_names, fold_id, png_dir, pdf_dir):
    mns = [m for m in MODEL_ORDER if m in data]
    ncols=min(4,len(mns)); nrows=(len(mns)+ncols-1)//ncols
    fig,axes=plt.subplots(nrows,ncols,figsize=(4.8*ncols,4.4*nrows),squeeze=False)
    flat=axes.flatten()
    for i,mn in enumerate(mns):
        ax=flat[i]; d=data[mn]
        cm=confusion_matrix(d["y_true"],d["y_pred"],labels=classes,normalize="true")
        im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1)
        ax.set_title(mn,fontweight="bold",fontsize=9)
        short=[c[:6] for c in class_names]
        ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(short,rotation=45,ha="right",fontsize=7)
        ax.set_yticklabels(short,fontsize=7)
        for r in range(len(classes)):
            for c2 in range(len(classes)):
                ax.text(c2,r,f"{cm[r,c2]:.2f}",ha="center",va="center",
                        color="white" if cm[r,c2]>0.5 else "black",fontsize=7)
        fig.colorbar(im,ax=ax,fraction=0.04,pad=0.03)
    for j in range(i+1,len(flat)): flat[j].set_visible(False)
    fig.suptitle(f"Confusion Matrices — {task} fold {fold_id}",fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"all_confusion_matrices.png"),
                  os.path.join(pdf_dir,"all_confusion_matrices.pdf"))

def plot_calibration(data, task, is_bin, fold_id, png_dir, pdf_dir):
    mns=[m for m in MODEL_ORDER if m in data]
    ncols=4; nrows=(len(mns)+ncols-1)//ncols
    fig,axes=plt.subplots(nrows,ncols,figsize=(4.5*ncols,4*nrows),squeeze=False)
    flat=axes.flatten()
    for i,mn in enumerate(mns):
        ax=flat[i]; d=data[mn]
        y_proba=d["y_proba"]; y_true=d["y_true"]
        if y_proba is None: ax.set_visible(False); continue
        try:
            if is_bin:
                prob=y_proba[:,1]; yt=y_true
            else:
                prob=y_proba.max(axis=1)
                yt=(y_true==y_proba.argmax(axis=1)).astype(int)
            frac,meanp=calibration_curve(yt,prob,n_bins=10)
            bs=brier_score_loss(yt,prob)
            ax.plot(meanp,frac,"o-",color=MODEL_COLORS.get(mn,"#607D8B"),ms=4)
            ax.plot([0,1],[0,1],"k--",lw=0.8)
            ax.set_title(f"{mn}\nBrier={bs:.4f}",fontsize=9)
        except: ax.set_title(f"{mn}\n(error)",fontsize=9)
        ax.set_xlabel("Mean pred prob."); ax.set_ylabel("Fraction positives")
    for j in range(i+1,len(flat)): flat[j].set_visible(False)
    fig.suptitle(f"Calibration — {task} fold {fold_id}",fontweight="bold",fontsize=12)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"calibration_curves.png"),
                  os.path.join(pdf_dir,"calibration_curves.pdf"))

def plot_threshold_cost(data, best_model, is_bin, fold_id, png_dir, pdf_dir):
    if not is_bin: return
    d=data.get(best_model)
    if d is None or d["y_proba"] is None: return
    y_true=d["y_true"]; prob1=d["y_proba"][:,1]
    thresholds=np.arange(0.01,0.99,0.01)
    precs,recs,f1s,specs,costs=[],[],[],[],[]
    from sklearn.metrics import precision_score,recall_score,f1_score
    for th in thresholds:
        yp=(prob1>=th).astype(int)
        cm=confusion_matrix(y_true,yp)
        tn,fp2,fn,tp=(cm.ravel() if cm.shape==(2,2) else [0,0,0,0])
        precs.append(precision_score(y_true,yp,zero_division=0))
        recs.append(recall_score(y_true,yp,zero_division=0))
        f1s.append(f1_score(y_true,yp,zero_division=0))
        specs.append(tn/(tn+fp2) if tn+fp2 else 0)
        costs.append(fn*FN_COST+fp2*FP_COST)
    best_f1_th=thresholds[np.argmax(f1s)]
    best_cost_th=thresholds[np.argmin(costs)]
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    ax=axes[0]
    ax.plot(thresholds,precs,label="Precision",color="#1565C0")
    ax.plot(thresholds,recs, label="Recall",   color="#B71C1C")
    ax.plot(thresholds,f1s,  label="F1",       color="#2E7D32",lw=2)
    ax.plot(thresholds,specs,label="Specificity",color="#F57F17",ls="--")
    ax.axvline(best_f1_th,  color="#2E7D32",ls=":", alpha=0.8,label=f"Best F1 th={best_f1_th:.2f}")
    ax.axvline(best_cost_th,color="#6A1B9A",ls="-.",alpha=0.8,label=f"Min cost th={best_cost_th:.2f}")
    ax.axvline(0.5,color="gray",ls="--",lw=0.7,label="Default 0.5")
    ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
    ax.set_title(f"Threshold Sensitivity — {best_model} fold {fold_id}",fontweight="bold")
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    ax2=axes[1]
    ax2.plot(thresholds,costs,color="#B71C1C",lw=2)
    ax2.axvline(best_cost_th,color="#6A1B9A",ls="--",
                label=f"Optimal th={best_cost_th:.2f}")
    min_c=min(costs); min_th=thresholds[np.argmin(costs)]
    ax2.annotate(f"Min cost={min_c:,.0f}\n@th={min_th:.2f}",
                 xy=(min_th,min_c),xytext=(min_th+0.1,min_c*1.05),
                 arrowprops=dict(arrowstyle="->"),fontsize=8)
    ax2.set_xlabel("Threshold")
    ax2.set_ylabel(f"Total cost (FN×{FN_COST}+FP×{FP_COST})")
    ax2.set_title("Cost-Sensitive Analysis",fontweight="bold")
    ax2.legend(fontsize=8)
    fig.suptitle(f"Threshold & Cost Analysis — fold {fold_id}",fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"threshold_cost_analysis.png"),
                  os.path.join(pdf_dir,"threshold_cost_analysis.pdf"))

def plot_perclass_heatmap(data, task, classes, class_names, is_bin, fold_id, png_dir, pdf_dir):
    if is_bin: return
    metrics=["precision","recall","f1-score"]
    heat_data={}
    for mn,d in data.items():
        rep=classification_report(d["y_true"],d["y_pred"],labels=classes,
                                  target_names=class_names,output_dict=True,zero_division=0)
        heat_data[mn]={cn:{m:rep[cn][m] for m in metrics} for cn in class_names if cn in rep}
    if not heat_data: return
    mns=[m for m in MODEL_ORDER if m in heat_data]
    fig,axes=plt.subplots(1,len(metrics),figsize=(5.5*len(metrics),5.5))
    for ax,met in zip(axes,metrics):
        mat=np.array([[heat_data[mn][cn][met] for cn in class_names] for mn in mns])
        im=ax.imshow(mat,cmap="RdYlGn",vmin=0.5,vmax=1.0,aspect="auto")
        ax.set_xticks(range(len(class_names)))
        ax.set_xticklabels(class_names,rotation=35,ha="right",fontsize=8)
        ax.set_yticks(range(len(mns))); ax.set_yticklabels(mns,fontsize=8)
        ax.set_title(met.capitalize(),fontweight="bold")
        for r in range(len(mns)):
            for c2 in range(len(class_names)):
                ax.text(c2,r,f"{mat[r,c2]:.2f}",ha="center",va="center",
                        fontsize=7.5,color="white" if mat[r,c2]<0.6 else "black")
        fig.colorbar(im,ax=ax,fraction=0.04,pad=0.03)
    fig.suptitle(f"Per-Class Heatmap — {task} fold {fold_id}",fontweight="bold",fontsize=12)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"per_class_heatmap.png"),
                  os.path.join(pdf_dir,"per_class_heatmap.pdf"))

def plot_error_analysis(csv_dir, task, class_names, fold_id, png_dir, pdf_dir):
    rows=[]
    for mn in MODEL_ORDER:
        mid=safe_name(mn)
        ep=os.path.join(csv_dir,f"errors_{mid}.csv")
        if not os.path.exists(ep): continue
        err=pd.read_csv(ep); err["model"]=mn; rows.append(err)
    if not rows: return
    all_err=pd.concat(rows,ignore_index=True)
    if "url" in all_err.columns:
        all_err["url_len"]=all_err["url"].astype(str).str.len()
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    ec=all_err.groupby("model").size().reindex(MODEL_ORDER).dropna()
    colors=[MODEL_COLORS.get(m,"#607D8B") for m in ec.index]
    axes[0].barh(ec.index,ec.values,color=colors)
    axes[0].set_xlabel("Error count")
    axes[0].set_title(f"Misclassification Count — fold {fold_id}",fontweight="bold")
    for i,(m,v) in enumerate(ec.items()):
        axes[0].text(v+20,i,f"{int(v):,}",va="center",fontsize=8)
    hgb_err=all_err[all_err["model"]=="HistGB"]
    if "tld" in hgb_err.columns and len(hgb_err):
        tld_c=hgb_err["tld"].value_counts().head(12)
        axes[1].barh(tld_c.index[::-1],tld_c.values[::-1],color=MODEL_COLORS["HistGB"])
        axes[1].set_xlabel("Error count")
        axes[1].set_title(f"HistGB Errors by TLD — fold {fold_id}",fontweight="bold")
        for i,(t,v) in enumerate(zip(tld_c.index[::-1],tld_c.values[::-1])):
            axes[1].text(v+1,i,str(int(v)),va="center",fontsize=8)
    else:
        axes[1].set_visible(False)
    fig.suptitle(f"Error Analysis — {task} fold {fold_id}",fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"error_analysis.png"),
                  os.path.join(pdf_dir,"error_analysis.pdf"))
    if "url_len" in all_err.columns:
        fig2,ax2=plt.subplots(figsize=(8,5))
        for mn in MODEL_ORDER:
            sub=all_err[all_err["model"]==mn]["url_len"]
            if len(sub): ax2.hist(sub,bins=40,alpha=0.45,label=mn,
                                  color=MODEL_COLORS.get(mn,"#607D8B"),density=True)
        ax2.set_xlabel("URL length (characters)"); ax2.set_ylabel("Density")
        ax2.set_title(f"URL Length of Misclassified URLs — fold {fold_id}",fontweight="bold")
        ax2.legend(fontsize=7)
        fig2.tight_layout()
        save_fig(fig2, os.path.join(png_dir,"error_url_length.png"),
                       os.path.join(pdf_dir,"error_url_length.pdf"))

# ============================================================
# 3. LOAD METADATA FROM MAIN EVAL RUN
# ============================================================
# Read the class names and metadata from saved fold 0 files
def get_meta(task):
    fold0_csv = os.path.join(DATA_DIR,f"journal_eval_{task}","fold0","csv")
    meta_p    = os.path.join(DATA_DIR,f"journal_eval_{task}","evaluation_metadata.json")
    if os.path.exists(meta_p):
        import json
        m = json.load(open(meta_p))
        return m["classes"], m["class_names"], m["task"]=="binary"
    # fallback: infer from class names
    if task=="binary":
        return [0,1], ["benign","malicious"], True
    return [0,1,2,3,4], ["benign","phishing","malware","scam","other_malicious"], False

# ============================================================
# 4. MAIN LOOP — generate plots for all folds
# ============================================================
print("="*70)
print("REGENERATING ALL FOLD PLOTS FROM SAVED .npy FILES")
print("="*70)

for TASK in TASKS:
    classes, class_names, is_bin = get_meta(TASK)
    classes = np.array(classes)
    print(f"\n  TASK: {TASK}  |  Classes: {class_names}")

    for FOLD in ALL_FOLDS:
        FOLD_DIR = os.path.join(DATA_DIR,f"journal_eval_{TASK}",f"fold{FOLD}")
        CSV_DIR  = os.path.join(FOLD_DIR,"csv")
        PNG_DIR  = os.path.join(FOLD_DIR,"figures_png")
        PDF_DIR  = os.path.join(FOLD_DIR,"figures_pdf")
        makedirs(PNG_DIR, PDF_DIR)

        if not os.path.exists(CSV_DIR):
            print(f"    fold {FOLD} — csv dir not found, skipping"); continue

        # check at least one model has predictions
        any_found = any(
            os.path.exists(os.path.join(CSV_DIR,f"y_true_{safe_name(mn)}.npy"))
            for mn in MODEL_ORDER)
        if not any_found:
            print(f"    fold {FOLD} — no .npy files found, skipping"); continue

        print(f"\n  ── Fold {FOLD} ──")
        data = load_fold_preds(CSV_DIR)
        print(f"    Loaded predictions for: {list(data.keys())}")

        # find best model by F1 for threshold plot
        best_by_f1 = "HistGB" if "HistGB" in data else list(data.keys())[0]

        plot_roc(data, TASK, classes, class_names, is_bin, FOLD, PNG_DIR, PDF_DIR)
        print(f"    ✓ ROC curves")

        plot_pr(data, TASK, classes, class_names, is_bin, FOLD, PNG_DIR, PDF_DIR)
        print(f"    ✓ PR curves")

        plot_det(data, TASK, is_bin, FOLD, PNG_DIR, PDF_DIR)
        if is_bin: print(f"    ✓ DET curves")

        plot_confusion_all(data, TASK, classes, class_names, FOLD, PNG_DIR, PDF_DIR)
        print(f"    ✓ Confusion matrices")

        plot_calibration(data, TASK, is_bin, FOLD, PNG_DIR, PDF_DIR)
        print(f"    ✓ Calibration curves")

        plot_threshold_cost(data, best_by_f1, is_bin, FOLD, PNG_DIR, PDF_DIR)
        if is_bin: print(f"    ✓ Threshold & cost analysis")

        plot_perclass_heatmap(data, TASK, classes, class_names, is_bin, FOLD, PNG_DIR, PDF_DIR)
        if not is_bin: print(f"    ✓ Per-class heatmap")

        plot_error_analysis(CSV_DIR, TASK, class_names, FOLD, PNG_DIR, PDF_DIR)
        print(f"    ✓ Error analysis")

        print(f"    All plots → {PNG_DIR}")

print(f"\n{'='*70}")
print("DONE — all fold plots generated.")
print(f"{'='*70}")
print(f"""
Files saved to:
  journal_eval_5class/fold0/ … fold4/
    figures_png/  roc_curves.png, pr_curves.png, all_confusion_matrices.png,
                  calibration_curves.png, per_class_heatmap.png,
                  error_analysis.png, error_url_length.png
  journal_eval_binary/fold0/ … fold4/
    figures_png/  + det_curves.png, threshold_cost_analysis.png
""")

REGENERATING ALL FOLD PLOTS FROM SAVED .npy FILES

  TASK: 5class  |  Classes: ['benign', 'phishing', 'malware', 'scam', 'other_malicious']

  ── Fold 0 ──
    Loaded predictions for: ['HistGB', 'LightGBM', 'XGBoost', 'RandomForest', 'ExtraTrees', 'LogisticRegression', 'ComplementNB']
    ✓ ROC curves
    ✓ PR curves
    ✓ Confusion matrices
    ✓ Calibration curves
    ✓ Per-class heatmap
    ✓ Error analysis
    All plots → /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/journal_eval_5class/fold0/figures_png

  ── Fold 1 ──
    Loaded predictions for: ['HistGB', 'LightGBM', 'XGBoost', 'RandomForest', 'ExtraTrees', 'LogisticRegression', 'ComplementNB']
    ✓ ROC curves
    ✓ PR curves
    ✓ Confusion matrices
    ✓ Calibration curves
    ✓ Per-class heatmap
    ✓ Error analysis
    All plots → /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/journal_eval_5class/fold1/figures_png

  ── Fold 2 ──
    Loaded predictions for: ['HistGB', 'LightGBM', 'XGBoost'